> **中文阅读版**：本文件是 `03_econml_dml.ipynb` 的 Markdown 翻译副本。代码、输出、图表和路径保持原样；专有名词和标识符保留英文。
# 动量因果效应的双重机器学习

**第 15 章：因果估计**
**Docker 镜像**：`ml4t`
**章节参考**：DML 理论与 ETF 因子应用见第 15.4 节

## 目的
**双重机器学习**（DML）在 ETF 面板上估计动量信号对未来收益的效应，同时调整波动率和 regime。该估计是本 Notebook 涉及的两件事之一。另一件是：面板会破坏所有具有“邻近”概念的校正——标准误、交叉验证折、置换块和时间子集都按行计数，除非明确要求不这样；而在堆叠面板上，一行对应的是不同的 ETF，而不是下一天。

## 学习目标
- LO1：解释混杂对因子回归的影响，以及调整能够恢复什么、不能恢复什么
- LO2：使用 EconML 为连续处理变量拟合 DML 估计器
- LO3：比较原始估计与调整后估计，并解读偏误方向
- LO4：构建按决策时点计数的折、标准误和置换
- LO5：解读反驳检验，包括一个无法反驳任何内容的检验

## 交叉引用
- **上游**：第 8 章（ETF 动量特征）
- **下游**：第 16 章（策略模拟）、第 19 章（风险管理）
- **相关**：[`02_dowhy_causal_graph`](02_dowhy_causal_graph.ipynb)（图方法）、[`04_dml_crypto_regime`](04_dml_crypto_regime.ipynb)（regime 效应）

## 方法论说明
参考 Chernozhukov 等 (2017) 与 de Prado (2018)：
- 来自 ml4t-diagnostics 的 **WalkForwardCV**，带 purging 和 embargo，基于面板的决策时点构建
- **Driscoll-Kraay 标准误**，在应用 Newey-West 核之前按决策时点聚合
- **实体内的块置换**，使安慰剂处理变量保持其持续性

**先决条件**：[`02_dowhy_causal_graph`](02_dowhy_causal_graph.ipynb)（用于了解 DoWhy 工作流），以及由特征流水线构建的 ETF 建模数据集

## 因果设计契约

| 要素                      | 本 Notebook                                                                                  |
|---------------------------|----------------------------------------------------------------------------------------------|
| 分析单元                  | 来自 ETF 建模面板的 ETF-日期行                                                               |
| 处理变量                  | `skip_recent_6_1` - 6/1 动量因子（连续）                                                      |
| 结果变量                  | `fwd_ret_21d` - 21 日未来收益                                                                 |
| 控制变量（EconML 中的 W） | `vol_21d`、`vol_126d`、`regime`、`yield_curve_slope` - 均为回看型、处理前变量                 |
| 效应调节变量（X）         | 本 Notebook 中无；目标为常数 ATE。X 槽位示例见 `04_dml_crypto_regime`                         |
| 识别假设                  | 在给定四个控制变量下满足可观测选择；处理前信息充分                                           |
| 主要失效模式              | 未观测混杂（情绪 / 宏观冲击）、nuisance 模型设定错误、若 CV 未按日期分组则出现面板-时间泄漏   |
| 估计目标                  | 调整后，6/1 动量每变动一个单位对 21 日未来收益的边际效应                                      |

## 设置

In [1]:
"""Double Machine Learning for Momentum Causal Effect - estimate causal effect of momentum on forward returns."""

import warnings

import numpy as np
import pandas as pd
from ml4t.diagnostic.splitters import WalkForwardCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge

from case_studies.utils.causal import (
    block_permute,
    empirical_permutation_p,
    manual_dml_timeseries,
)
from utils.modeling import load_modeling_dataset
from utils.reproducibility import set_global_seeds

# scikit-learn repeats a notice, once per nuisance fit, that a frame carrying feature names
# was fitted and a bare array predicted; EconML does that internally. Convergence and
# numerical warnings stay visible.
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.utils.validation")

# Statsmodels for HAC standard errors
import statsmodels.api as sm
from econml.dml import LinearDML
from statsmodels.regression.linear_model import OLS

## 配置

In [2]:
# Configuration - readers can modify these
CASE_STUDY_ID = "etfs"
PRIMARY_LABEL = "fwd_ret_21d"
MAX_SYMBOLS = 0
CV_FOLDS = 5
MAX_SAMPLES = 50000  # Temporal subsample if dataset too large
SEED = 42

# Cross-validation parameters for WalkForwardCV
FORWARD_HORIZON = 21  # 21-day forward returns
LABEL_HORIZON = FORWARD_HORIZON  # Purge overlapping samples
EMBARGO_PCT = 0.01  # 1% embargo after test set

# Refutation test parameters
N_PLACEBO_PERMUTATIONS = 100
BLOCK_SIZE = 21  # Block size for permutation (match forward horizon)

In [3]:
set_global_seeds(SEED)
print(f"Seed: {SEED}")

Seed: 42


## 1. 从建模流水线加载 ETF 特征

我们使用 `load_modeling_dataset()` 加载预计算特征（第 8 章）、时间特征（第 9 章）和标签，并完成连接、可直接用于分析。仅使用真实数据——没有合成数据回退方案。如果建模数据集缺失，Notebook 会以清晰错误显式失败，而不是静默切换到合成替代数据，从而发布无法区分的数字。

In [4]:
# A load failure is fatal rather than a fallback to synthetic data.
mds = load_modeling_dataset(CASE_STUDY_ID, PRIMARY_LABEL, max_symbols=MAX_SYMBOLS)

treatment_col = "skip_recent_6_1"
outcome_col = mds.label_col
confounder_cols = ["vol_21d", "vol_126d", "regime", "yield_curve_slope"]
entity_col = mds.entity_cols[0]

available = set(mds.dataset.columns)
missing = [c for c in [treatment_col, outcome_col] + confounder_cols if c not in available]
if missing:
    raise RuntimeError(
        f"Required columns missing from modeling dataset "
        f"{CASE_STUDY_ID}/{PRIMARY_LABEL}: {missing}. "
        f"Available features: {mds.feature_names[:20]}... "
        f"Set ML4T_DATA_PATH and rebuild the Ch8 features pipeline for case "
        f"study '{CASE_STUDY_ID}'."
    )

# Convert to pandas for sklearn/econml, sorted by date
analysis_cols = [mds.date_col] + mds.entity_cols + [treatment_col, outcome_col] + confounder_cols
df = (
    mds.dataset.select([c for c in analysis_cols if c in available])
    .drop_nulls()
    .sort(mds.date_col)
    .to_pandas()
)
# Temporal subsample if too large: take the most recent N unique dates so the
# subsample never cuts through a cross-section. `df.iloc[-MAX_SAMPLES:]` would
# slice at row level on a stacked panel and leave a fragmented final date.
if len(df) > MAX_SAMPLES:
    rows_per_date = df.groupby(mds.date_col).size().median()
    n_dates = int(np.ceil(MAX_SAMPLES / max(rows_per_date, 1)))
    keep_dates = df[mds.date_col].drop_duplicates().iloc[-n_dates:]
    df = df[df[mds.date_col].isin(keep_dates)].reset_index(drop=True)
    print(f"Taking most recent {n_dates} dates ({len(df):,} rows) from {len(keep_dates):,} dates")

print(f"Analysis data: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Date range: {df[mds.date_col].min()} to {df[mds.date_col].max()}")
print(f"Treatment: {treatment_col}, Outcome: {outcome_col}")

Taking most recent 556 dates (52,162 rows) from 556 dates
Analysis data: 52,162 rows × 8 columns
Date range: 2023-09-14 00:00:00 to 2025-12-01 00:00:00
Treatment: skip_recent_6_1, Outcome: fwd_ret_21d


### 面板键，以及下文为何都需要它们

这里的一行是一个 ETF *和* 一个日期，数据框按日期排序，因此连续行通常是同一天的不同 ETF，而不是同一 ETF 的连续日期。本 Notebook 中所有具有“邻近”概念的校正——标准误、交叉验证折、置换块、时间子集——都必须按决策时点计数，而不是按行计数。在这个数据框上按行计数，度量的是某一天横截面的一个切片，却把它称作三周。

下面两个数组承载该信息，上述四个位置都会使用它们。

In [5]:
decision_times = df[mds.date_col].to_numpy()
entities = df[entity_col].to_numpy()

unique_dates = np.sort(pd.unique(decision_times))
date_position = pd.Series(np.arange(len(unique_dates)), index=unique_dates)
row_date_position = date_position.reindex(decision_times).to_numpy()

print(
    f"{len(df):,} rows over {len(unique_dates):,} decision times "
    f"and {df[entity_col].nunique()} entities "
    f"({len(df) / len(unique_dates):.0f} rows per decision time)"
)


def panel_folds(n_splits):
    """Walk-forward folds built over decision times, then expanded to panel rows.

    WalkForwardCV counts `label_horizon` and the embargo in the positions it is handed. Fed
    the panel's rows it would purge a fraction of one date; fed the ordered unique dates it
    purges the 21 trading days the forward return actually spans. The row indices come back
    by membership, so no fold boundary cuts through a cross-section.
    """
    splitter = WalkForwardCV(
        n_splits=n_splits,
        label_horizon=LABEL_HORIZON,
        embargo_pct=EMBARGO_PCT,
        expanding=True,
    )
    folds = []
    for train_dates, test_dates in splitter.split(np.arange(len(unique_dates)).reshape(-1, 1)):
        folds.append(
            (
                np.flatnonzero(np.isin(row_date_position, train_dates)),
                np.flatnonzero(np.isin(row_date_position, test_dates)),
            )
        )
    return folds

52,162 rows over 556 decision times and 97 entities (94 rows per decision time)


## 2. 未调整斜率

将未来收益对动量做 OLS 回归，不包含其他变量。它是调整后估计所对照的基准，并且按构造即受混杂：波动率和收益率曲线 regime 都会影响动量信号，也会影响未来收益，因此其斜率既承载动量的贡献，也承载它们的贡献。这会朝哪个方向推动斜率，是一个经验问题，下面的比较会给出答案。

### 面板数据采用哪种稳健标准误

重叠的 21 日收益率使同一只 ETF 的连续观测值相关，通常的答案是使用 Newey-West 标准误，并将带宽设为标签期限。在这个数据框上，这个答案被应用到了错误的轴上。`cov_type="HAC"` 沿行方向运行其核，而这里的 21 行只相当于某一天横截面的五分之一，因此该校正会把同一天的不同 ETF 当作连续多天来处理。

**Driscoll-Kraay** 才是适合面板数据的版本。它先按决策时间聚合回归得分，再对该时间序列应用 Newey-West 核，从而既能抵抗重叠造成的序列相关，也能抵抗 ETF 在给定日期共享的任何因素。statsmodels 通过 `cov_type="hac-groupsum"` 和 `time` 参数实现它，这正是 `case_studies/utils/causal.py` 为每个案例研究所做的相同调用。

In [6]:
y = df[outcome_col].values
X_with_const = sm.add_constant(df[[treatment_col]])

ols_iid = OLS(y, X_with_const).fit()
naive_estimate = float(ols_iid.params.iloc[1])
se_iid = float(ols_iid.bse.iloc[1])

# Driscoll-Kraay: the kernel runs over decision times, with the bandwidth at the label
# horizon because a 21-day forward return overlaps for 20 of every 21 days.
HAC_LAGS = FORWARD_HORIZON
time_codes = pd.factorize(decision_times, sort=False)[0]
ols_dk = ols_iid.get_robustcov_results(
    cov_type="hac-groupsum",
    time=time_codes,
    maxlags=HAC_LAGS,
    use_correction="hac",
    df_correction=False,
)
se_hac = float(np.sqrt(np.asarray(ols_dk.cov_params())[1, 1]))
t_stat_hac = naive_estimate / se_hac

naive_ci = (naive_estimate - 1.96 * se_hac, naive_estimate + 1.96 * se_hac)

print("=" * 60)
print("NAIVE ESTIMATE (ignoring confounders)")
print("=" * 60)
print(f"Coefficient: {naive_estimate:.6f}")
print(f"Standard Error (IID): {se_iid:.6f}")
print(f"Standard Error (Driscoll-Kraay): {se_hac:.6f}")
print(f"SE Inflation (DK/IID): {se_hac / se_iid:.2f}x")
print(f"95% CI (Driscoll-Kraay): [{naive_ci[0]:.6f}, {naive_ci[1]:.6f}]")
print(f"t-statistic (Driscoll-Kraay): {t_stat_hac:.2f}")

NAIVE ESTIMATE (ignoring confounders)
Coefficient: -0.035896
Standard Error (IID): 0.002104
Standard Error (Driscoll-Kraay): 0.027912
SE Inflation (DK/IID): 13.26x
95% CI (Driscoll-Kraay): [-0.090603, 0.018811]
t-statistic (Driscoll-Kraay): -1.29


## 3. 可观测混杂因素调整下的双重机器学习估计

DML 使用三步正交化过程：

1. **用混杂因素预测结果**：$\hat{Y} = g(X)$ → 残差 $\tilde{Y} = Y - \hat{Y}$
2. **用混杂因素预测处理**：$\hat{T} = m(X)$ → 残差 $\tilde{T} = T - \hat{T}$
3. **用残差处理回归残差结果**：$\tilde{Y} \sim \theta \tilde{T}$

系数 $\theta$ 是在保持以下假设的前提下对处理效应的*正交化*估计：所指定的控制变量捕捉了相关的处理前混杂变化。DML 降低了对冗余模型误差的敏感性，但其本身并不能解决未观测混杂、联立性、干扰或坏控制偏差。

折来自 `WalkForwardCV`，而不是 `KFold` 或 `TimeSeriesSplit`，因为对前瞻性标签进行交叉拟合需要两件这两者都不提供的东西。**净化（Purging）**会删除 21 日标签窗口与测试窗口重叠的训练行；**禁闭期（embargo）**在测试窗口之后留出间隔，使自相关不会带回下一个训练集（Chernozhukov et al. 2017, de Prado 2018）。

混杂因素放入 EconML 的 `W` 槽位，该槽位保存用于残差化的控制变量。`X` 槽位用于效应修饰变量，即允许处理效应沿其变化的变量。将普通混杂因素放入 `X` 仍会产生单一的 ATE，因此表面上没有任何问题，但这会为 `04_dml_crypto_regime` 养成错误的习惯；在该案例中，两个槽位承载不同的变量，而这一区别决定了模型估计的是什么。

In [7]:
Y = df[outcome_col].to_numpy()
T = df[treatment_col].to_numpy()
W = df[confounder_cols].to_numpy()  # controls used for residualization

# Walk-forward folds over decision times, expanded to rows (see `panel_folds` above).
cv = panel_folds(CV_FOLDS)
print(
    f"Using WalkForwardCV over decision times (label_horizon={LABEL_HORIZON} trading days, "
    f"embargo={EMBARGO_PCT:.1%})"
)
for i, (train_idx, test_idx) in enumerate(cv):
    print(f"  fold {i}: train {len(train_idx):,} rows, test {len(test_idx):,} rows")

dml = LinearDML(
    model_y=GradientBoostingRegressor(n_estimators=50, max_depth=3, random_state=SEED),
    model_t=GradientBoostingRegressor(n_estimators=50, max_depth=3, random_state=SEED),
    cv=cv,
    random_state=SEED,
)

dml.fit(Y, T, W=W)

dml_estimate = float(dml.ate())
dml_ci_lower, dml_ci_upper = (float(v) for v in dml.ate_interval(alpha=0.05))
bias = naive_estimate - dml_estimate
bias_pct = 100 * bias / abs(dml_estimate) if dml_estimate != 0 else float("inf")

Using WalkForwardCV over decision times (label_horizon=21 trading days, embargo=1.0%)
  fold 0: train 6,816 rows, test 8,556 rows
  fold 1: train 15,384 rows, test 8,556 rows
  fold 2: train 23,940 rows, test 8,597 rows
  fold 3: train 32,516 rows, test 8,648 rows
  fold 4: train 41,164 rows, test 9,024 rows


In [8]:
print("\n" + "=" * 60)
print("DOUBLE ML ESTIMATE (controlling for confounders)")
print("=" * 60)
print(f"Average Treatment Effect: {dml_estimate:.6f}")
if not np.isnan(dml_ci_lower):
    print(f"95% CI: [{dml_ci_lower:.6f}, {dml_ci_upper:.6f}]")
else:
    print(
        "95% CI: unavailable (EconML inference returned NaN - "
        "likely numerically degenerate first-stage residualization)"
    )

print("\n" + "=" * 60)
print("COMPARISON: Naive vs DML")
print("=" * 60)
print(f"Naive estimate: {naive_estimate:.6f}")
print(f"DML estimate:   {dml_estimate:.6f}")
print(f"Bias (Naive - DML): {bias:.6f}")
print(f"Bias percentage: {bias_pct:.1f}%")

if abs(naive_estimate) > abs(dml_estimate):
    print("\n-> The adjusted effect is smaller: the controls were inflating the raw slope")
else:
    print("\n-> The adjusted effect is larger: the controls were masking part of the slope")


DOUBLE ML ESTIMATE (controlling for confounders)
Average Treatment Effect: -0.058517
95% CI: [-0.065232, -0.051802]

COMPARISON: Naive vs DML
Naive estimate: -0.035896
DML estimate:   -0.058517
Bias (Naive - DML): 0.022621
Bias percentage: 38.7%

-> The adjusted effect is larger: the controls were masking part of the slope


**解读**：朴素 OLS 估计同时捕捉了因果动量效应和由共享混杂因素（波动率、regime）引起的虚假相关。DML 的正交化通过在估计最终系数之前对处理变量和结果变量都关于混杂因素做残差化，从而消除这种混杂。

偏误的方向揭示了混杂结构。如果朴素效应的幅度小于 DML 估计，则混杂因素*掩盖*了真实效应——例如，高波动率同时降低动量和收益。如果朴素效应更大，则混杂因素*夸大*了表观预测能力。

**它对下游的影响**：由原始斜率推导出的头寸规模会继承混杂因素对它的一切贡献。当波动率和 regime 未被单独对冲时，应基于调整后的估计来确定规模，而两者之间的差距就是原始斜率所承载的部分。

## 4. 手工组装相同的估计

`case_studies/utils/causal.py` 中的 `manual_dml_timeseries` 显式执行这三个步骤：在带禁闭期的前向滚动折上交叉拟合两个冗余模型，取残差，再将一个残差对另一个残差做回归。传入 `groups` 时，它会按决策时间构建这些折并报告 Driscoll-Kraay 标准误；不传入时，它会按行完成这两件事。

In [9]:
# Use shared manual DML implementation
dml_result = manual_dml_timeseries(
    df[outcome_col].values,
    df[treatment_col].values,
    df[confounder_cols].values,
    n_folds=CV_FOLDS,
    embargo=LABEL_HORIZON,
    groups=decision_times,
    horizon=FORWARD_HORIZON,
)

manual_ate = dml_result["theta"]
manual_se_iid = dml_result["se_iid"]
manual_se_hac = dml_result["se_hac"]
manual_t_hac = dml_result["t_stat_hac"]
manual_ci = (manual_ate - 1.96 * manual_se_hac, manual_ate + 1.96 * manual_se_hac)

print("\n" + "=" * 60)
print("MANUAL DML WITH WALK-FORWARD CV + EMBARGO")
print("=" * 60)
print(f"ATE estimate: {manual_ate:.6f}")
print(f"Standard Error (IID): {manual_se_iid:.6f}")
print(f"Standard Error (HAC): {manual_se_hac:.6f}")
print(f"SE Inflation (HAC/IID): {manual_se_hac / manual_se_iid:.2f}x")
print(f"95% CI (HAC): [{manual_ci[0]:.6f}, {manual_ci[1]:.6f}]")
print(f"t-statistic (Driscoll-Kraay): {manual_t_hac:.2f}")


MANUAL DML WITH WALK-FORWARD CV + EMBARGO
ATE estimate: -0.049251
Standard Error (IID): 0.004014
Standard Error (HAC): 0.026538
SE Inflation (HAC/IID): 6.61x
95% CI (HAC): [-0.101264, 0.002763]
t-statistic (Driscoll-Kraay): -1.86


## 5. 反驳检验

**对因果有效性至关重要**：我们使用反驳检验来验证 DML 估计。

1. **时间安慰剂**：将 Y 对 T 的*领先项*（lead）做回归（若不存在反向因果，应约为 0）
2. **分块置换**：按块打乱处理变量以保持自相关
3. **子集稳定性**：检查效应在不同时间子集上是否稳定

In [10]:
print("\n" + "=" * 60)
print("REFUTATION TESTS")
print("=" * 60)

# The lead is taken within symbol; a row-level shift would mix ETFs at the boundary.
print("\n1. TEMPORAL PLACEBO TEST (lead of treatment)")
df_placebo = df.sort_values([entity_col, mds.date_col]).copy()
df_placebo["treatment_lead"] = df_placebo.groupby(entity_col)[treatment_col].shift(-FORWARD_HORIZON)
# Back to date order before the fit. The shift needed entity-major rows; the folds need
# every row of a decision time adjacent, and `manual_dml_timeseries` rejects groups that
# are not sorted and contiguous rather than silently splitting one date across two folds.
df_placebo = (
    df_placebo.dropna(subset=["treatment_lead", outcome_col])
    .sort_values(mds.date_col, kind="stable")
    .reset_index(drop=True)
)

if len(df_placebo) > 100:
    placebo_result = manual_dml_timeseries(
        df_placebo[outcome_col].values,
        df_placebo["treatment_lead"].values,
        df_placebo[confounder_cols].values,
        n_folds=CV_FOLDS,
        embargo=LABEL_HORIZON,
        groups=df_placebo[mds.date_col].to_numpy(),
        horizon=FORWARD_HORIZON,
    )
    placebo_effect = placebo_result["theta"]

    print(f"   Lead treatment effect (DML): {placebo_effect:.6f}")
    print(f"   Original DML effect:         {manual_ate:.6f}")
    print(f"   Ratio |lead / original|:     {abs(placebo_effect) / abs(manual_ate):.3f}")
else:
    placebo_effect = None
    print("   Insufficient data for placebo test")


REFUTATION TESTS

1. TEMPORAL PLACEBO TEST (lead of treatment)


   Lead treatment effect (DML): -0.075183
   Original DML effect:         -0.049251
   Ratio |lead / original|:     1.527


**解读时间安慰剂。**这里该检验返回的比率接近 1，这并非反向因果的证据。处理变量是一个动量信号，在 21 个交易日内几乎不变，因此按期限平移的安慰剂几乎与真实处理变量相同，并会重现相似的效应。对于如此持续的处理变量，平移信号安慰剂无法反驳任何东西。

下面的分块置换才是能够反驳的检验，因为它在保持处理变量自身持续性不变的同时，打破了处理变量与结果变量之间的对齐——但前提是分块沿时间构建。

### 沿时间分块，而非沿行分块

当给 `block_permute` 传入一个裸数组时，它会按传入的位置来计 `BLOCK_SIZE`。在这个数据框上，这大约是一天横截面（cross-section）的五分之一，因此一个“块”只是单日的一小撮 ETF。对这些进行置换会破坏该检验本应保留的序列依赖（serial dependence）；这正是 `block_permute` 自己的 docstring 所警告的 iid 打乱，会让安慰剂检验（placebo test）过于容易通过。

传入 `groups` 和 `units` 后，它会在每只 ETF 内部进行置换；此时 `BLOCK_SIZE` 按该 ETF 自身有序交易日计数，一个块就是名称所承诺的三周。

每个安慰剂也运行被检验的估计器，并使用相同的折数和 embargo。用三折拟合的安慰剂与用五折拟合的估计相比，是在不同数量的交叉拟合行上使用不同的估计器；此时原假设（null）的中心会落在该差异所决定的位置，而不是落在效应不存在所决定的位置。观测统计量本身在两种设置之间也会移动。

In [11]:
# Test 2: Block Permutation Test (uses shared block_permute)
print(f"\n2. BLOCK PERMUTATION TEST ({N_PLACEBO_PERMUTATIONS} permutations)")
placebo_effects = []
placebo_t_stats = []
permutation_failures = 0
T_original = df[treatment_col].values
rng = np.random.default_rng(SEED)

for i in range(N_PLACEBO_PERMUTATIONS):
    T_permuted = block_permute(
        T_original,
        BLOCK_SIZE,
        rng=rng,
        groups=decision_times,
        units=entities,
    )

    df_perm = df.copy()
    df_perm[treatment_col] = T_permuted

    try:
        perm_result = manual_dml_timeseries(
            df_perm[outcome_col].values,
            df_perm[treatment_col].values,
            df_perm[confounder_cols].values,
            n_folds=CV_FOLDS,  # the estimate's own setting; see the markdown above
            embargo=LABEL_HORIZON,
            groups=decision_times,
            horizon=FORWARD_HORIZON,
        )
        if np.isfinite(perm_result["t_stat_hac"]):
            placebo_t_stats.append(perm_result["t_stat_hac"])
            placebo_effects.append(perm_result["theta"])
        else:
            permutation_failures += 1
    except Exception as exc:
        permutation_failures += 1
        print(f"   Permutation {i} failed: {type(exc).__name__}: {exc}")

PERMUTATION_MIN_SUCCESS = max(10, int(0.5 * N_PLACEBO_PERMUTATIONS))
print(f"   Permutations: {len(placebo_effects)} successful, {permutation_failures} failed")
if len(placebo_effects) < PERMUTATION_MIN_SUCCESS:
    raise RuntimeError(
        f"Block-permutation placebo: only {len(placebo_effects)} successful "
        f"runs (need ≥{PERMUTATION_MIN_SUCCESS}); refuter cannot be trusted."
    )


2. BLOCK PERMUTATION TEST (100 permutations)


   Permutations: 100 successful, 0 failed


### 按 t 统计量比较，而非按效应大小比较

对处理变量（treatment）进行置换，也使其脱离控制变量。第二阶段将残差化后的结果变量对残差化后的处理变量做回归，因此控制变量所能解释的部分会从分母中消失；被置换的处理变量不再由这些控制变量解释，其残差方差更大，于是无论是否真有可发现的东西，安慰剂效应都会机械地小于观测效应。拿原始效应与该分布比较，会报告出标准误并不支持的显著性。每次置换的 t 统计量都除以其自身的标准误，因此尺度被抵消，只剩处理变量与结果变量之间的对齐关系。

p 值是在安慰剂分布中至少与观测统计量一样极端的比例，并带有加一校正（plus-one correction），因为观测统计量本身也是该分布可能产生的一次抽样。若不进行该校正，当某次运行中没有任何安慰剂达到观测统计量时就会报告为零，而任何有限次置换都无法确立零。有 n 次抽样时，最小可报告值为 1 / (n + 1)，并在旁边作为下限（floor）打印出来。它不是错误发现率（false discovery rate）；这个量过去曾被这样称呼。

**抵消尺度并不能换来什么。** 它消除了一种已知偏差，但并不能让检验变得校准（calibrated）。置换检验只有在原假设下被置换的标签可交换（exchangeable）时才有效；而一个由持续存在的混杂因素（confounder）所预测的处理变量并不满足这一点：打乱在破坏效应的同时也破坏了混杂结构，因此本身有偏的估计会远离其自身的置换原假设，检验报告的正是这段距离。在十二个合成面板（synthetic panel）上测量——真实效应恰好为零且存在持续 AR(1) 混杂因素——学生化（studentized）比较在十二个中仍有五个在传统 5% 水平上拒绝，而原始效应比较为十一个。`10_case_study_insights` 载有该测量结果。请把下面的计数理解为与打乱结果的比较，而不是显著性水平。

In [12]:
if len(placebo_t_stats) > 10:
    placebo_mean = np.mean(placebo_t_stats)
    placebo_std = np.std(placebo_t_stats)
    z_score = (manual_t_hac - placebo_mean) / placebo_std if placebo_std > 0 else np.inf
    # The plus-one correction is why the floor below is 1 / (n + 1); see the markdown above.
    permutation_p = empirical_permutation_p(np.asarray(placebo_t_stats), manual_t_hac)

    print(f"   Placebo t mean: {placebo_mean:.4f}")
    print(f"   Placebo t std:  {placebo_std:.4f}")
    print(f"   Observed t (Driscoll-Kraay): {manual_t_hac:.4f}")
    print(f"   Z-score vs placebo: {z_score:.2f}")
    print(
        f"   Permutation p-value: {permutation_p:.4f} (floor {1 / (len(placebo_t_stats) + 1):.4f})"
    )

    print(
        f"   Placebo draws at least as extreme: "
        f"{int(round(permutation_p * (len(placebo_t_stats) + 1))) - 1} of {len(placebo_t_stats)}"
    )
    print(
        f"   Placebo effect spread {np.std(placebo_effects):.6f} against a Driscoll-Kraay "
        f"standard error of {manual_se_hac:.6f}; the comparison above is on t-statistics, "
        f"which holds whatever the ratio of those two turns out to be"
    )
else:
    print("   Insufficient successful permutations")
    z_score = None
    permutation_p = None

   Placebo t mean: 1.7379
   Placebo t std:  1.2094
   Observed t (Driscoll-Kraay): -1.8559
   Z-score vs placebo: -2.97
   Permutation p-value: 0.4752 (floor 0.0099)
   Placebo draws at least as extreme: 47 of 100
   Placebo effect spread 0.023095 against a Driscoll-Kraay standard error of 0.026538; the comparison above is on t-statistics, which holds whatever the ratio of those two turns out to be


**z 分数和 p 值可能不一致，而计数才是可靠的那个。** z 分数衡量观测 t 统计量距离安慰剂 *均值* 有多少个安慰剂标准差，这是关于一个以安慰剂所在位置为中心的正态分布的陈述。置换 p 值则统计有多少次安慰剂抽样达到其量级。当安慰剂分布并非以零附近为中心时，二者回答的是不同问题，只有计数才是关于检验实际构建的原假设的陈述。在读 z 分数之前，先读上面打印的计数，以及其旁边的均值和标准差。

**这个原假设并非以零为中心，这一点值得审视，而不只是报告。** 若置换实现了“无关联”，安慰剂统计量应当落在零附近。两个显而易见的解释经不起检验：按 symbol 对拟合残差去均值后，安慰剂估计仍然为正；按 date 去均值也一样，因此无论是保留的 symbol 间成分，还是保留的 date 间成分，都无法解释这一点。真正的原因在此尚未确定。在确定之前，计数才是应读取的统计量——它问的是被置换的处理变量有多频繁达到观测统计量，无论分布中心在哪里，这都仍是一个合理问题——而 z 分数以安慰剂标准差衡量与那个中心的距离，是关于一个该检验尚未解释的分布的陈述。

这两半是在决策时点而非某一行处切分的，因此二者都不包含横截面的碎片。靠近顶部处的子样本保留完整日期，也是同样的原因。

In [13]:
print("\n3. SUBSET STABILITY TEST (temporal halves)")
midpoint_date = unique_dates[len(unique_dates) // 2]
df_first_half = df[decision_times < midpoint_date]
df_second_half = df[decision_times >= midpoint_date]

if len(df_first_half) > 100 and len(df_second_half) > 100:
    result_first = manual_dml_timeseries(
        df_first_half[outcome_col].values,
        df_first_half[treatment_col].values,
        df_first_half[confounder_cols].values,
        n_folds=3,
        groups=df_first_half[mds.date_col].to_numpy(),
        horizon=FORWARD_HORIZON,
    )
    result_second = manual_dml_timeseries(
        df_second_half[outcome_col].values,
        df_second_half[treatment_col].values,
        df_second_half[confounder_cols].values,
        n_folds=3,
        groups=df_second_half[mds.date_col].to_numpy(),
        horizon=FORWARD_HORIZON,
    )

    effect_first = result_first["theta"]
    se_first = result_first["se_hac"]
    effect_second = result_second["theta"]
    se_second = result_second["se_hac"]

    print(f"   First half effect:  {effect_first:.6f} (SE: {se_first:.6f})")
    print(f"   Second half effect: {effect_second:.6f} (SE: {se_second:.6f})")

    diff = abs(effect_first - effect_second)
    diff_se = np.sqrt(se_first**2 + se_second**2)
    diff_z = diff / diff_se if diff_se > 0 else 0

    print(f"   Difference: {diff:.6f} (z = {diff_z:.2f})")

    print(f"   First half: {len(df_first_half):,} rows before {midpoint_date}")
    print(f"   Second half: {len(df_second_half):,} rows from {midpoint_date}")
else:
    print("   Insufficient data for subset test")


3. SUBSET STABILITY TEST (temporal halves)


   First half effect:  -0.062793 (SE: 0.042257)
   Second half effect: -0.017201 (SE: 0.032803)
   Difference: 0.045592 (z = 0.85)
   First half: 26,079 rows before 2024-10-22T00:00:00.000
   Second half: 26,083 rows from 2024-10-22T00:00:00.000


## 6. nuisance 模型敏感性

DML 结果取决于 nuisance 模型的质量。我们来比较不同选择。

In [14]:
results = []

nuisance_models = [
    ("Linear", Ridge(alpha=1.0), Ridge(alpha=1.0)),
    (
        "GBM (shallow)",
        GradientBoostingRegressor(n_estimators=50, max_depth=2, random_state=SEED),
        GradientBoostingRegressor(n_estimators=50, max_depth=2, random_state=SEED),
    ),
    (
        "GBM (deep)",
        GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=SEED),
        GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=SEED),
    ),
]

cv_sensitivity = panel_folds(3)

for name, model_y, model_t in nuisance_models:
    dml_test = LinearDML(model_y=model_y, model_t=model_t, cv=cv_sensitivity, random_state=SEED)
    dml_test.fit(Y, T, W=W)
    ate = float(dml_test.ate())
    results.append({"Nuisance Model": name, "ATE Estimate": ate})

sensitivity_df = pd.DataFrame(results)
print("\n" + "=" * 60)
print("NUISANCE MODEL SENSITIVITY")
print("=" * 60)
print(sensitivity_df.to_string(index=False))
# A ratio of magnitudes reports 1.00x, which reads as no spread at all, when the
# specifications straddle zero symmetrically - the one case where the nuisance choice
# changes the conclusion rather than its size. The signed range cannot do that.
lo = float(sensitivity_df["ATE Estimate"].min())
hi = float(sensitivity_df["ATE Estimate"].max())
print(
    f"\nAcross {len(sensitivity_df)} nuisance specifications the estimate spans "
    f"{lo:.6f} to {hi:.6f}, a range of {hi - lo:.6f} in {outcome_col} units"
)
print(f"  Specifications agreeing in sign: {'all' if lo * hi > 0 else 'not all'}")


NUISANCE MODEL SENSITIVITY
Nuisance Model  ATE Estimate
        Linear     -0.063726
 GBM (shallow)     -0.082061
    GBM (deep)     -0.086313

Across 3 nuisance specifications the estimate spans -0.086313 to -0.063726, a range of 0.022587 in fwd_ret_21d units
  Specifications agreeing in sign: all


## 7. 结果摘要

In [15]:
print("\n" + "=" * 60)
print("CHAPTER 15 NOTEBOOK RESULTS: 03_econml_dml.py")
print("=" * 60)

results_summary = {
    "naive_estimate": naive_estimate,
    "naive_se_iid": se_iid,
    "naive_se_hac": se_hac,
    "naive_ci_lower": naive_ci[0],
    "naive_ci_upper": naive_ci[1],
    "manual_dml_estimate": manual_ate,
    "manual_dml_se_iid": manual_se_iid,
    "manual_dml_se_hac": manual_se_hac,
}

if dml_estimate is not None:
    results_summary["econml_dml_estimate"] = dml_estimate
    results_summary["econml_dml_ci_lower"] = dml_ci_lower
    results_summary["econml_dml_ci_upper"] = dml_ci_upper
    results_summary["bias_from_confounding"] = bias
    results_summary["bias_pct"] = bias_pct

if z_score is not None:
    results_summary["placebo_z_score"] = z_score
if permutation_p is not None:
    results_summary["permutation_p"] = permutation_p

for key, value in results_summary.items():
    if isinstance(value, float):
        print(f"{key}: {value:.6f}")
    else:
        print(f"{key}: {value}")


CHAPTER 15 NOTEBOOK RESULTS: 03_econml_dml.py
naive_estimate: -0.035896
naive_se_iid: 0.002104
naive_se_hac: 0.027912
naive_ci_lower: -0.090603
naive_ci_upper: 0.018811
manual_dml_estimate: -0.049251
manual_dml_se_iid: 0.004014
manual_dml_se_hac: 0.026538
econml_dml_estimate: -0.058517
econml_dml_ci_lower: -0.065232
econml_dml_ci_upper: -0.051802
bias_from_confounding: 0.022621
bias_pct: 38.657063
placebo_z_score: -2.971589
permutation_p: 0.475248


## 关键方法论要点

### 决定估计是否有意义的四个选择
1. **带 purging 与 embargo 的交叉拟合（cross-fitting）**，按决策时点而非按行进行。
   Purging 会删除标签与测试窗口重叠的训练行；embargo 在其后增加一段缓冲（Chernozhukov et al. 2017, de Prado 2018）。
2. **Driscoll-Kraay 标准误**，因此序列相关校正沿时间方向施加，同一日期内的截面依赖被吸收，而不会被计为额外观测。
3. **实体内的分块置换**，因此安慰剂分布由保留其持续性的处理构建。
4. **nuisance 模型扫描**，因为点估计取决于第一阶段，而不同设定间的差距才是该发现的诚实宽度。

### 同一效应的三个区间

该 notebook 对同一数量给出三种不确定性陈述，而它们并不一致。EconML 的 `ate_interval` 将残差化观测视为独立抽样。手动 DML 路径报告 Driscoll-Kraay 标准误，它按决策时点聚合，并吸收 ETF 在某一交易日共同承受的部分。置换检验则将估计值与通过在每只 ETF 内打乱处理构建的原假设进行比较。

它们之间的差距并非其中任何一个的缺陷。这是面板数据付出的代价：区间的好坏取决于它对“什么独立”的刻画，而 52,000 个 ETF-日并不是 52,000 个独立观测。

In [16]:
se_inflation = se_hac / se_iid

print("Quantitative Findings")
print("-" * 40)
print(f"SE inflation, Driscoll-Kraay over iid: {se_inflation:.2f}x")
if dml_estimate is not None:
    direction = "larger" if abs(naive_estimate) > abs(dml_estimate) else "smaller"
    print(
        f"Adjustment moves the slope by {abs(bias_pct):.1f}% - the unadjusted estimate is "
        f"{direction} in magnitude"
    )
    if not np.isnan(dml_ci_lower):
        print(
            f"EconML DML: {dml_estimate:.6f}, iid 95% CI "
            f"[{dml_ci_lower:.6f}, {dml_ci_upper:.6f}], width {dml_ci_upper - dml_ci_lower:.6f}"
        )
    else:
        print(f"EconML DML: {dml_estimate:.6f} (interval unavailable)")
print(
    f"Manual DML:  {manual_ate:.6f}, Driscoll-Kraay 95% CI "
    f"[{manual_ci[0]:.6f}, {manual_ci[1]:.6f}], width {manual_ci[1] - manual_ci[0]:.6f}"
)
print(
    f"Naive OLS:   {naive_estimate:.6f}, Driscoll-Kraay 95% CI [{naive_ci[0]:.6f}, {naive_ci[1]:.6f}]"
)
if z_score is not None:
    print(f"Placebo z-score (on t-statistics): {z_score:.2f}")
if permutation_p is not None:
    print(f"Permutation p-value: {permutation_p:.4f}")

Quantitative Findings
----------------------------------------
SE inflation, Driscoll-Kraay over iid: 13.26x
Adjustment moves the slope by 38.7% - the unadjusted estimate is smaller in magnitude
EconML DML: -0.058517, iid 95% CI [-0.065232, -0.051802], width 0.013430
Manual DML:  -0.049251, Driscoll-Kraay 95% CI [-0.101264, 0.002763], width 0.104027
Naive OLS:   -0.035896, Driscoll-Kraay 95% CI [-0.090603, 0.018811]
Placebo z-score (on t-statistics): -2.97
Permutation p-value: 0.4752


## 关键要点

1. **DML 是调整后的估计，而不是免费午餐**。只有在维持假设下，正交化系数才可解释为因果效应：处理前控制变量充分、正值性（positivity）成立、无干扰，并且指定的 DAG 正确。

2. **稳健标准误必须与数据的轴相匹配。** 重叠的 21 日标签会使 iid 标准误过小，而在堆叠面板上，普通 Newey-West 校正沿错误的轴修正：其滞后跨越截面。Driscoll-Kraay 先按决策时点聚合，这正是上文打印的标准误膨胀所度量的内容。

3. **手动 DML 在概念上与 EconML 一致，在数值上则不同。** 点估计会随 nuisance 模型的灵活性而变化，第 6 节的扫描展示了变化幅度。Neyman 正交性保证的是对 nuisance 误差的一阶敏感性，而不是不同 nuisance 选择之间的一致性。

4. **证伪检验应以其名称所声称的单位来计数。** 分块置换只有在块沿实体自身交易日排列时才保留自相关；在展平面板上，同一调用置换的是某一天截面的一片，而这正是该检验要避免的 iid 打乱。前向滚动折和时间对半也同样如此。

5. **混杂偏差的符号是结果，而不是规则。** 在这里，控制变量在一个方向上改变动量斜率；在 `02_dowhy_causal_graph` 的加密资金费率溢价中，它们则朝另一个方向改变。上文打印的比较说明了本数据上发生了什么，而定论的是打印出的数字，而不是这句话。